# MA3632 — Workshop 3: Data Transformation

This workshop accompanies Lecture 3. Part A covers scaling and normalisation.
Part B covers categorical encoding. Part C covers distribution transformation.
Part D ties everything together inside a `sklearn` Pipeline, enforcing the
train/test discipline discussed in Section 4 of the lecture.

---

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

# Load the working dataset once; all parts draw from this split.
# Falls back to a locally generated synthetic dataset with the same columns
# if the network fetch is unavailable, so the workshop runs offline.
try:
    raw = fetch_california_housing(as_frame=True)
    df = raw.frame.copy()
except Exception as e:
    print(f"California Housing fetch unavailable ({type(e).__name__}); using synthetic fallback.")
    n_synth = 3000
    rng_synth = np.random.default_rng(seed=42)
    med_inc    = rng_synth.gamma(shape=5.0, scale=0.8, size=n_synth)
    house_age  = rng_synth.uniform(1, 52, size=n_synth)
    ave_rooms  = rng_synth.normal(5.5, 1.2, size=n_synth).clip(min=1)
    ave_bedrms = ave_rooms * rng_synth.uniform(0.15, 0.35, size=n_synth)
    population = rng_synth.gamma(shape=3.0, scale=400, size=n_synth)
    ave_occup  = rng_synth.normal(3.0, 0.7, size=n_synth).clip(min=1)
    latitude   = rng_synth.uniform(32.5, 42.0, size=n_synth)
    longitude  = rng_synth.uniform(-124.3, -114.3, size=n_synth)
    med_house_val = (
        0.5 * med_inc + 0.01 * (52 - house_age) - 0.05 * ave_occup
        + rng_synth.normal(0, 0.4, size=n_synth)
    ).clip(min=0.15, max=5.0)
    df = pd.DataFrame({
        "MedInc": med_inc, "HouseAge": house_age, "AveRooms": ave_rooms,
        "AveBedrms": ave_bedrms, "Population": population, "AveOccup": ave_occup,
        "Latitude": latitude, "Longitude": longitude, "MedHouseVal": med_house_val,
    })

X = df.drop(columns="MedHouseVal")
y = df["MedHouseVal"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0
)

print(f"Training set:  {X_train.shape[0]} rows")
print(f"Test set:      {X_test.shape[0]} rows")
print(f"Features:      {list(X.columns)}")

---

## Part A — Scaling and normalisation

### A1. Visual motivation

Before scaling, the features span very different ranges. This causes distance-based
and gradient-based algorithms to treat high-magnitude features as more important.

In [ ]:
print("Feature ranges (training set):")
print(X_train.agg(["min", "max", "std"]).round(3).to_string())

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(14, 6))
axes = axes.flatten()

for ax, col in zip(axes, X_train.columns):
    ax.hist(X_train[col], bins=40, color="steelblue", edgecolor="none", alpha=0.85)
    ax.set_title(col, fontsize=9)
    ax.tick_params(labelsize=7)

fig.suptitle("California Housing — raw feature distributions (training set)", fontsize=10)
plt.tight_layout()
plt.show()

### A2. Min-max scaling

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler_mm = MinMaxScaler()
scaler_mm.fit(X_train)               # parameters estimated on training data only

X_train_mm = pd.DataFrame(
    scaler_mm.transform(X_train),
    columns=X_train.columns, index=X_train.index
)
X_test_mm = pd.DataFrame(
    scaler_mm.transform(X_test),
    columns=X_test.columns, index=X_test.index
)

print("Scaled training set — min and max per feature:")
print(X_train_mm.agg(["min", "max"]).round(4).to_string())
print()
print("Scaled test set — note values can fall outside [0, 1]:")
print(X_test_mm.agg(["min", "max"]).round(4).to_string())

Test values can fall outside $[0,1]$ when they lie outside the training range.
This is expected behaviour, not an error; the scaler must not be re-fitted on the
test set to prevent it.

### A3. Standardisation

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler_std = StandardScaler()
scaler_std.fit(X_train)

X_train_std = pd.DataFrame(
    scaler_std.transform(X_train),
    columns=X_train.columns, index=X_train.index
)
X_test_std = pd.DataFrame(
    scaler_std.transform(X_test),
    columns=X_test.columns, index=X_test.index
)

print("Scaled training set — mean and std per feature:")
print(X_train_std.agg(["mean", "std"]).round(4).to_string())

### A4. Robust scaling

In [ ]:
from sklearn.preprocessing import RobustScaler

scaler_rob = RobustScaler()
scaler_rob.fit(X_train)

X_train_rob = pd.DataFrame(
    scaler_rob.transform(X_train),
    columns=X_train.columns, index=X_train.index
)
X_test_rob = pd.DataFrame(
    scaler_rob.transform(X_test),
    columns=X_test.columns, index=X_test.index
)

print("Robust-scaled training set — median and IQR per feature:")
print(X_train_rob.agg(["median", lambda x: x.quantile(0.75) - x.quantile(0.25)])
      .rename(index={"<lambda>": "IQR"}).round(4).to_string())

### A5. Comparison

In [ ]:
# Compare the three scalers on AveRooms, which has a noticeable right tail
col = "AveRooms"

fig, axes = plt.subplots(1, 3, figsize=(13, 4))

for ax, data, title in zip(
    axes,
    [X_train_mm[col], X_train_std[col], X_train_rob[col]],
    ["Min-max", "Standardisation", "Robust"]
):
    ax.hist(data, bins=40, color="steelblue", edgecolor="none", alpha=0.85)
    ax.axvline(data.mean(), color="firebrick", linewidth=1.2,
               linestyle="--", label=f"mean={data.mean():.2f}")
    ax.set_title(f"{title} — {col}")
    ax.set_xlabel("Scaled value")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

# Quantify the effect of the outliers
print(f"AveRooms max (training):         {X_train[col].max():.1f}")
print(f"After min-max — max:             {X_train_mm[col].max():.4f}")
print(f"After standardisation — max:     {X_train_std[col].max():.4f}")
print(f"After robust scaling — max:      {X_train_rob[col].max():.4f}")

### A6. Quantile transformation

Min-max, standardisation, and robust scaling are all affine: they shift and rescale
but cannot change the *shape* of a distribution. The quantile transform maps each
observation to its empirical rank (optionally passed through the standard normal
quantile function), forcing a target marginal shape at the cost of distorting
relative distances between observations.


In [ ]:
from sklearn.preprocessing import QuantileTransformer

qt_uniform = QuantileTransformer(output_distribution="uniform", random_state=0)
qt_normal  = QuantileTransformer(output_distribution="normal", random_state=0)

qt_uniform.fit(X_train)   # empirical CDF estimated on training data only
qt_normal.fit(X_train)

X_train_qu = pd.DataFrame(qt_uniform.transform(X_train), columns=X_train.columns, index=X_train.index)
X_train_qn = pd.DataFrame(qt_normal.transform(X_train), columns=X_train.columns, index=X_train.index)

col = "AveRooms"
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for ax, data, title in zip(
    axes,
    [X_train_mm[col], X_train_std[col], X_train_rob[col], X_train_qn[col]],
    ["Min-max", "Standardisation", "Robust", "Quantile (Gaussian)"]
):
    ax.hist(data, bins=40, color="seagreen" if title.startswith("Quantile") else "steelblue",
            edgecolor="none", alpha=0.85)
    ax.set_title(title, fontsize=9)

fig.suptitle(f"{col} under four scaling strategies (training set)", fontsize=10)
plt.tight_layout()
plt.show()

print(f"Skewness of {col}:")
print(f"  raw:                {X_train[col].skew():.3f}")
print(f"  robust-scaled:      {X_train_rob[col].skew():.3f}")
print(f"  quantile (Gaussian): {X_train_qn[col].skew():.3f}")


**Exercise A.** Construct a small array `v = [2, 4, 6, 8, 10, 200]` and apply all
three scalers manually using only NumPy (no sklearn). Verify that your results match
sklearn by passing `v` through each fitted scaler. Then explain in one sentence why
the robust-scaled value of `200` is so much larger than its min-max-scaled value.

---

## Part B — Encoding categorical variables

We use a dataset with both nominal and ordinal categorical features to illustrate the
difference between the two encoding strategies.

In [ ]:
# Construct a small employee dataset with deliberate categorical structure
np.random.seed(1)
n = 200

dept      = np.random.choice(["Engineering", "Marketing", "Finance", "HR"], n,
                              p=[0.4, 0.25, 0.25, 0.1])
education = np.random.choice(["Secondary", "Undergraduate", "Postgraduate"], n,
                              p=[0.2, 0.55, 0.25])
years_exp = np.random.randint(0, 20, n)

# Salary: depends on department and education level (ground truth we can check later)
base = {"Engineering": 60000, "Finance": 55000, "Marketing": 48000, "HR": 42000}
edu_bonus = {"Secondary": 0, "Undergraduate": 5000, "Postgraduate": 12000}

salary = np.array([
    base[d] + edu_bonus[e] + years_exp[i] * 1200 + np.random.normal(0, 4000)
    for i, (d, e) in enumerate(zip(dept, education))
])

df_emp = pd.DataFrame({
    "department": dept,
    "education":  education,
    "years_exp":  years_exp,
    "salary":     salary.round(0)
})

print(df_emp.head(8))
print("\nDepartment counts:", df_emp["department"].value_counts().to_dict())
print("Education counts: ", df_emp["education"].value_counts().to_dict())

### B1. One-hot encoding (nominal)

In [ ]:
from sklearn.preprocessing import OneHotEncoder

enc_ohe = OneHotEncoder(drop="first", sparse_output=False)
enc_ohe.fit(df_emp[["department"]])

dept_encoded = pd.DataFrame(
    enc_ohe.transform(df_emp[["department"]]),
    columns=enc_ohe.get_feature_names_out(["department"])
)

print("One-hot encoded department (drop='first', reference = Engineering):")
print(dept_encoded.head(8))
print(f"\nShape: {dept_encoded.shape}  — {dept_encoded.shape[1]} columns for 4 categories")

In [ ]:
# Demonstrate the dummy variable trap: what happens with drop=None
enc_full = OneHotEncoder(drop=None, sparse_output=False)
dept_full = enc_full.fit_transform(df_emp[["department"]])

# The sum of all k columns equals 1 for every row
row_sums = dept_full.sum(axis=1)
print("Sum of all k indicator columns, first 5 rows:", row_sums[:5])
print("This constant-sum constraint makes the design matrix rank-deficient.")

### B2. Ordinal encoding

In [ ]:
from sklearn.preprocessing import OrdinalEncoder

edu_order = [["Secondary", "Undergraduate", "Postgraduate"]]

enc_ord = OrdinalEncoder(categories=edu_order)
enc_ord.fit(df_emp[["education"]])

edu_encoded = enc_ord.transform(df_emp[["education"]])
df_emp["education_enc"] = edu_encoded.astype(int)

print("Ordinal encoding of education level:")
print(df_emp[["education", "education_enc"]].drop_duplicates()
      .sort_values("education_enc").to_string(index=False))

In [ ]:
# Check that the encoding respects the ordering in the salary relationship
print("Mean salary by education level:")
print(df_emp.groupby("education_enc")["salary"].mean().round(0))

# Compare to a naive integer encoding that ignores the order
naive_map = {"Secondary": 0, "Undergraduate": 2, "Postgraduate": 1}  # wrong order
df_emp["education_naive"] = df_emp["education"].map(naive_map)

print("\nMean salary by naive (mis-ordered) encoding:")
print(df_emp.groupby("education_naive")["salary"].mean().round(0))
print("The monotone relationship is broken by the wrong ordering.")

### B3. Target encoding (high cardinality)

In [ ]:
# Simulate a high-cardinality feature: postcode (50 categories)
np.random.seed(2)
postcodes = [f"PC{i:03d}" for i in range(50)]
df_emp["postcode"] = np.random.choice(postcodes, n)

# One-hot encoding would produce 49 columns for 50 categories
# Target encoding produces 1 column

# Compute leave-one-out target encoding to avoid leakage.
# Vectorised: precompute each group's sum and count once, then adjust per row,
# rather than re-filtering the whole frame inside a per-row loop.
def target_encode_loo(df, col, target):
    """Leave-one-out target encoding."""
    group_sum = df.groupby(col)[target].transform("sum")
    group_n   = df.groupby(col)[target].transform("count")
    total_sum = df[target].sum()
    total_n   = len(df)
    loo_mean = np.where(
        group_n > 1,
        (group_sum - df[target]) / (group_n - 1),
        total_sum / total_n
    )
    return loo_mean

df_emp["postcode_enc"] = target_encode_loo(df_emp, "postcode", "salary")

print("Target encoding sample (first 6 rows):")
print(df_emp[["postcode", "salary", "postcode_enc"]].head(6).round(1))
print(f"\nStandard deviation of encoded values: {df_emp['postcode_enc'].std():.1f}")
print("One column instead of 49 indicator columns.")

### B4. Frequency encoding and the hashing trick

Target encoding (B3) requires a target variable and, even with leave-one-out
correction, carries some residual leakage risk for small categories. Frequency
encoding avoids the target entirely; the hashing trick handles cardinalities too
large even for target or frequency encoding to be practical.


In [ ]:
# Frequency encoding: uses no information about the target at all
freq_map = df_emp["postcode"].value_counts(normalize=True)
df_emp["postcode_freq"] = df_emp["postcode"].map(freq_map)

print("Frequency encoding sample (first 6 rows):")
print(df_emp[["postcode", "postcode_freq"]].head(6).round(4))
print(f"\nUnique frequency values: {df_emp['postcode_freq'].nunique()} "
      f"(out of {df_emp['postcode'].nunique()} postcodes) — one column, no target used.")

# The hashing trick: map an unbounded number of categories to a fixed number of buckets
from sklearn.feature_extraction import FeatureHasher

hasher = FeatureHasher(n_features=8, input_type="string")
unique_codes = df_emp["postcode"].unique()
hashed_unique = hasher.transform([[c] for c in unique_codes]).toarray()
n_patterns = len(set(map(tuple, hashed_unique)))

print(f"\n{len(unique_codes)} distinct postcodes hash into {hashed_unique.shape[1]} buckets, "
      f"producing {n_patterns} distinct bucket patterns — some postcodes collide.")


**Exercise B.** Build a full design matrix for a linear regression of `salary` on
`department` (one-hot, drop first), `education` (ordinal), and `years_exp`.
Fit `sklearn.linear_model.LinearRegression` and print the coefficients. Do the signs
and magnitudes agree with the ground-truth salary structure used to generate the data?

---

## Part C — Distribution transformation

### C1. Identifying skewness

In [ ]:
from scipy import stats

# AveRooms and Population in the California Housing data are right-skewed
cols_skew = ["AveRooms", "Population", "MedInc"]

print("Skewness of selected features (training set):")
for col in cols_skew:
    sk = stats.skew(X_train[col])
    print(f"  {col:<15} {sk:.3f}")

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, col in zip(axes, cols_skew):
    ax.hist(X_train[col], bins=50, color="steelblue", edgecolor="none", alpha=0.85)
    ax.set_title(col)
    ax.set_xlabel("Raw value")
plt.suptitle("Right-skewed features — raw distributions", fontsize=10)
plt.tight_layout()
plt.show()

### C2. Log transformation

In [ ]:
# Apply log1p (log(x+1)) to handle any zero values safely
log_transformed = X_train[cols_skew].apply(np.log1p)

print("Skewness after log1p transformation:")
for col in cols_skew:
    sk_raw = stats.skew(X_train[col])
    sk_log = stats.skew(log_transformed[col])
    print(f"  {col:<15}  raw: {sk_raw:.3f}   log1p: {sk_log:.3f}")

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, col in zip(axes, cols_skew):
    ax.hist(log_transformed[col], bins=50, color="darkorange",
            edgecolor="none", alpha=0.85)
    ax.set_title(f"log1p({col})")
    ax.set_xlabel("Transformed value")
plt.suptitle("Features after log1p transformation", fontsize=10)
plt.tight_layout()
plt.show()

### C3. Box-Cox transformation

In [ ]:
from sklearn.preprocessing import PowerTransformer

# PowerTransformer with method='box-cox' requires strictly positive values
# Fit on training data; apply to test data with the same lambda
pt = PowerTransformer(method="box-cox", standardize=False)

col = "AveRooms"
pt.fit(X_train[[col]])
print(f"Estimated Box-Cox lambda for {col}: {pt.lambdas_[0]:.4f}")

X_train_bc = pt.transform(X_train[[col]])
X_test_bc  = pt.transform(X_test[[col]])

sk_raw = stats.skew(X_train[col])
sk_bc  = stats.skew(X_train_bc.ravel())
print(f"Skewness — raw: {sk_raw:.3f}   Box-Cox: {sk_bc:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(X_train[col], bins=50, color="steelblue", edgecolor="none", alpha=0.85)
axes[0].set_title(f"{col} — raw  (skew={sk_raw:.2f})")
axes[1].hist(X_train_bc.ravel(), bins=50, color="darkorange",
             edgecolor="none", alpha=0.85)
axes[1].set_title(f"{col} — Box-Cox λ={pt.lambdas_[0]:.3f}  (skew={sk_bc:.2f})")
plt.tight_layout()
plt.show()

### C4. Transforming the target variable

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error

# Fit a linear regression on a single feature with and without log-transforming y
feature = "MedInc"
X_tr = X_train[[feature]].values
X_te = X_test[[feature]].values
y_tr = y_train.values
y_te = y_test.values

# Model 1: raw target
m_raw = LinearRegression().fit(X_tr, y_tr)
pred_raw = m_raw.predict(X_te)
rmse_raw = root_mean_squared_error(y_te, pred_raw)

# Model 2: log-transformed target (predict on log scale, back-transform)
m_log = LinearRegression().fit(X_tr, np.log(y_tr))
pred_log = np.exp(m_log.predict(X_te))
rmse_log = root_mean_squared_error(y_te, pred_log)

print(f"RMSE — raw target:           {rmse_raw:.4f}")
print(f"RMSE — log-transformed target: {rmse_log:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for ax, preds, title in zip(
    axes,
    [pred_raw, pred_log],
    ["Raw target", "Log-transformed target"]
):
    ax.scatter(y_te, preds, alpha=0.2, s=8, color="steelblue")
    lim = [min(y_te.min(), preds.min()), max(y_te.max(), preds.max())]
    ax.plot(lim, lim, "r--", linewidth=1)
    ax.set_xlabel("Actual")
    ax.set_ylabel("Predicted")
    ax.set_title(title)

plt.tight_layout()
plt.show()

**Exercise C.** Apply the Yeo-Johnson transformation (`method='yeo-johnson'` in
`PowerTransformer`) to `Population`. Unlike Box-Cox, Yeo-Johnson handles zero and
negative values. Compare the skewness before and after. Then explain why applying
any monotone transformation to features has no effect on a decision tree's predictions,
but may affect a linear regression model's fit.

---

## Part D — The sklearn Pipeline

All transformation parameters must be estimated on training data and applied to test
data using those training estimates. A `Pipeline` enforces this automatically and
ensures the same procedure is replicated inside cross-validation folds.

### D1. Building the pipeline

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score

# For illustration we add a synthetic nominal feature to the housing data
X_demo = X_train.copy()
X_demo_test = X_test.copy()

np.random.seed(0)
regions = np.random.choice(["coastal", "inland", "valley"], len(X_demo), p=[0.4,0.4,0.2])
X_demo["region"]      = regions
X_demo_test["region"] = np.random.choice(["coastal", "inland", "valley"],
                                          len(X_demo_test), p=[0.4,0.4,0.2])

numeric_features = list(X_train.columns)
nominal_features = ["region"]

numeric_transformer = Pipeline([
    ("scaler", StandardScaler()),
])

nominal_transformer = Pipeline([
    ("ohe", OneHotEncoder(drop="first", sparse_output=False, handle_unknown="ignore")),
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("nom", nominal_transformer, nominal_features),
])

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model",        Ridge(alpha=1.0)),
])

print("Pipeline steps:")
for name, step in pipeline.steps:
    print(f"  {name}: {step}")

In [ ]:
# Fit on training data — all transformation parameters estimated here
pipeline.fit(X_demo, y_train)

# Evaluate on test data — transformations applied with training parameters
pred_pipe = pipeline.predict(X_demo_test)
rmse_pipe = root_mean_squared_error(y_test, pred_pipe)
print(f"Pipeline RMSE on test set: {rmse_pipe:.4f}")

# Cross-validation — pipeline re-fits transformers inside each fold
cv_scores = cross_val_score(pipeline, X_demo, y_train,
                             cv=5, scoring="neg_root_mean_squared_error")
print(f"5-fold CV RMSE: {-cv_scores.mean():.4f}  (std {cv_scores.std():.4f})")

### D2. Demonstrating leakage

In [ ]:
# Contrast the correct pipeline approach with the leaky approach

# CORRECT: fit scaler on training data only
scaler_correct = StandardScaler()
scaler_correct.fit(X_train)
X_tr_correct = scaler_correct.transform(X_train)
X_te_correct = scaler_correct.transform(X_test)

# LEAKY: fit scaler on full dataset before splitting
X_all = pd.concat([X_train, X_test])
scaler_leaky = StandardScaler()
scaler_leaky.fit(X_all)
X_tr_leaky = scaler_leaky.transform(X_train)
X_te_leaky = scaler_leaky.transform(X_test)

m_correct = Ridge(alpha=1.0).fit(X_tr_correct, y_train)
m_leaky   = Ridge(alpha=1.0).fit(X_tr_leaky,   y_train)

rmse_correct = root_mean_squared_error(y_test, m_correct.predict(X_te_correct))
rmse_leaky   = root_mean_squared_error(y_test, m_leaky.predict(X_te_leaky))

print(f"RMSE — correct (train-only scaler): {rmse_correct:.4f}")
print(f"RMSE — leaky   (full-data scaler):  {rmse_leaky:.4f}")
print()
print("Difference in scaler means (leaky vs correct):")
diff = scaler_leaky.mean_ - scaler_correct.mean_
for col, d in zip(X_train.columns, diff):
    print(f"  {col:<20} {d:+.4f}")

On this particular dataset the RMSE difference from leakage is small, because the
training and test distributions are similar. In general --- when the test set is from
a different time period, region, or population --- leakage can produce a large
optimistic bias in the reported test performance, leading to a model that appears
better than it is.

---

## Take-home exercises

**Exercise 1.** Replace the `StandardScaler` in the pipeline with a `RobustScaler`
and rerun cross-validation. Does the CV RMSE improve? Examine the distribution of
`AveRooms` in the training set and argue whether robust scaling is likely to help
for this feature.

**Exercise 2.** Add a `PowerTransformer` (Box-Cox) step to the numeric transformer
in the pipeline. Check that the pipeline still runs correctly and report the new
CV RMSE. Which features benefit most from the transformation?

**Exercise 3.** (Written, no code.) Suppose you are building a model to predict
customer churn. Your dataset contains a `country` column with 80 distinct values and
a roughly uniform distribution across them. Discuss the trade-offs between one-hot
encoding and target encoding for this column, including the leakage risk of each.

**Exercise 4.** Fit the full pipeline (preprocessor + Ridge) using 10-fold
cross-validation and plot the distribution of the 10 RMSE values as a histogram.
What does the spread of this distribution tell you about the stability of the
model's performance?

---